In [3]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 106.5 MB/s eta 0:00:00


In [11]:
%%writefile app.py
import os, re, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib
import streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# ============================================================
# 0. THEME / PAGE CONFIG  (from Login_Page.ipynb)
# ============================================================
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#0d0b1e", "bg_sidebar": "#150f30", "bg_card": "#1a1442", "bg_card_alt": "#2a2160",
    "text_main": "#e8e6f7", "text_heading": "#ffffff", "text_muted": "#a29bd4",
    "accent": "#ff5da2", "accent_hover": "#ff3d8f", "accent_text": "#0d0b1e",
    "border": "#3d2f7a", "border_light": "#4b3a94", "success": "#34d399", "danger": "#ff6b6b"
}

# ============================================================
# 1. SECRETS  -- FIX: nothing hardcoded, everything from env
#    (In Colab, set these first: os.environ['JWT_SECRET']=userdata.get('JWT_SECRET') etc.
#     See the launcher cell at the bottom of this file's companion notebook cell.)
# ============================================================
JWT_SECRET     = os.environ.get("JWT_SECRET")
SENDER_EMAIL   = os.environ.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD")
ADMIN_EMAIL    = os.environ.get("ADMIN_EMAIL", "infosys@ai")
ADMIN_PASSWORD = os.environ.get("ADMIN_PASSWORD", "admin@123")  # only used to seed the admin row once
OTP_EXPIRY_MINUTES = 5

if not JWT_SECRET:
    st.error("❌ JWT_SECRET not found. Add it to Colab Secrets and load it into os.environ before running.")
    st.stop()

# ============================================================
# 2. CSS (Neo-brutalist, from Login_Page.ipynb)
# ============================================================
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');

    html, body, .stApp {{
        background: radial-gradient(circle at top left, #2a2160 0%, {COLORS['bg_main']} 55%, #0a0818 100%) !important;
        font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important;
    }}
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: none !important;
        border-radius: 10px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 0 4px 14px rgba(255,93,162,0.5) !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: #fff !important; color: #fff !important; stroke: #fff !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_muted']} !important; }}

    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{
    background-color: {COLORS['bg_card']} !important; border: 1px solid {COLORS['border']} !important; border-radius: 12px !important;
}}
div[data-baseweb="input"]:focus-within {{
    border-color: {COLORS['accent']} !important; box-shadow: 0 0 0 3px rgba(255,93,162,0.25) !important;
}}

/* FIX: force the actual <input> element (not just its wrapper) to be dark, with visible white text */
div[data-baseweb="input"] input, textarea, div[data-baseweb="select"] span {{
    background-color: transparent !important;
    color: {COLORS['text_heading']} !important;
    -webkit-text-fill-color: {COLORS['text_heading']} !important;
    caret-color: {COLORS['text_heading']} !important;
}}
input::placeholder, textarea::placeholder {{
    color: {COLORS['text_muted']} !important;
    opacity: 1 !important;
}}
/* FIX: kill Chrome/Edge autofill's forced light-yellow background */
input:-webkit-autofill {{
    -webkit-box-shadow: 0 0 0 30px {COLORS['bg_card']} inset !important;
    -webkit-text-fill-color: {COLORS['text_heading']} !important;
}}
    div[data-testid="stButton"] button {{
        background: linear-gradient(135deg, {COLORS['accent']}, #7b5cff) !important; color: #fff !important;
        border: none !important; border-radius: 12px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; box-shadow: 0 6px 18px rgba(123,92,255,0.35) !important;
        width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        transform: translateY(-2px) !important; box-shadow: 0 10px 24px rgba(255,93,162,0.45) !important;
    }}

    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 1px solid {COLORS['border']} !important; }}

    .pn-card {{
        background: rgba(26,20,66,0.65); backdrop-filter: blur(10px);
        border: 1px solid {COLORS['border_light']}; border-radius: 16px; padding: 24px;
        box-shadow: 0 8px 32px rgba(0,0,0,0.35);
    }}
</style>
""", unsafe_allow_html=True)

# ============================================================
# 3. DB HELPERS  (schema from Login_Page.ipynb, kept as single source of truth)
# ============================================================
def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    if not conn.execute("SELECT id FROM users WHERE email=?", (ADMIN_EMAIL,)).fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", ADMIN_EMAIL, hash_txt(ADMIN_PASSWORD),
                      "What is your pet name?", hash_txt("admin")))

# ============================================================
# 4. VALIDATION HELPERS  -- FIX: Step 8 (email format + password complexity)
# ============================================================
EMAIL_RE = re.compile(r"^[A-Za-z]{2,}[A-Za-z0-9._%+-]*@[A-Za-z]{2,}\.[A-Za-z]{2,}$")
PASSWORD_RE = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[^A-Za-z0-9]).{8,}$")

def valid_email(e): return bool(EMAIL_RE.match(e))
def valid_password(p): return bool(PASSWORD_RE.match(p))

PASSWORD_HINT = "Min 8 characters, with at least one uppercase, one lowercase, one number and one symbol."

# ============================================================
# 5. JWT SESSION HELPERS
# ============================================================
def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception: return None

# ============================================================
# 6. OTP HELPERS  (from OTP_New.ipynb, adapted to reuse the same JWT_SECRET)
# ============================================================
def generate_otp(): return f"{secrets.randbelow(900000) + 100000}"

def make_otp_token(email, otp):
    payload = {"sub": email, "otp_hash": hash_txt(otp), "type": "password_reset_otp",
               "iat": datetime.datetime.utcnow(), "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp":
            return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]): return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError:
        return False, f"⚠️ This OTP code expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception:
        return False, "Invalid or corrupted verification token."

def send_otp_email(to_email, otp):
    if not SENDER_EMAIL or not EMAIL_PASSWORD:
        return False, "Gmail sender address / app password not configured in secrets."
    msg = MIMEMultipart('alternative')
    msg['From'] = f"Infosys Support <{SENDER_EMAIL}>"
    msg['To'] = to_email
    msg['Subject'] = "Infosys Portal - Verification Code"
    msg['Date'] = formatdate(localtime=True)
    msg['Message-ID'] = make_msgid()
    msg['Reply-To'] = SENDER_EMAIL

    text_body = f"Your verification code for Infosys Portal is: {otp}\nThis code will expire in {OTP_EXPIRY_MINUTES} minutes.\nIf you did not request this code, please ignore this email."
    html_body = f"""
    <html><body style="font-family:Arial,sans-serif;background:#f9fcfc;padding:20px;">
      <div style="max-width:500px;margin:0 auto;background:#fff;border:2px solid #272343;border-radius:12px;padding:30px;text-align:center;">
        <div style="color:#272343;font-size:20px;font-weight:bold;margin-bottom:15px;">Infosys Portal Verification</div>
        <div style="color:#4a5568;font-size:15px;margin-bottom:20px;">We received a request to reset your password for <b>{to_email}</b>. Use the code below:</div>
        <div style="background:#ffd803;color:#272343;font-size:28px;font-weight:bold;letter-spacing:5px;padding:15px 20px;border:2px solid #272343;border-radius:8px;display:inline-block;margin:10px 0;">{otp}</div>
        <div style="color:#4a5568;font-size:15px;margin-top:15px;">This code expires in <b>{OTP_EXPIRY_MINUTES} minutes</b>.</div>
      </div></body></html>"""
    msg.attach(MIMEText(text_body, 'plain'))
    msg.attach(MIMEText(html_body, 'html'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(SENDER_EMAIL, EMAIL_PASSWORD)
        s.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        s.quit()
        return True, "Email sent successfully!"
    except Exception as e:
        return False, f"SMTP Error: {str(e)}"

# ============================================================
# 7. SESSION STATE
# ============================================================
for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None),
             ("otp_jwt", None), ("otp_stage", None)]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# 8. PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:

        # ---------------- LOGIN ----------------
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
                if not email or not pwd:
                    st.error("⚠️ Please fill in both fields.")
                else:
                    with get_db() as c: r = c.execute("SELECT password_hash FROM users WHERE email=?", (email,)).fetchone()
                    if r and check_txt(pwd, r[0]):
                        st.session_state.token = make_jwt(email); navigate("Dashboard")
                    else:
                        st.error("❌ Invalid credentials.")  # generic - never says which field was wrong
            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

        # ---------------- SIGNUP ----------------
        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.caption(PASSWORD_HINT)
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account →", use_container_width=True):
                if not uname or not email or not pwd or not confirm_pwd or not sa:
                    st.error("⚠️ Please fill all fields.")
                elif not valid_email(email):
                    st.error("❌ Enter a valid email address (e.g. ab@cd.ef).")
                elif not valid_password(pwd):
                    st.error(f"❌ Weak password. {PASSWORD_HINT}")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                                      (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.success("✅ Account created! Please sign in.")
                        time.sleep(1)
                        navigate("Login")   # JWT only issued at Login (Step 7)
                    except sqlite3.IntegrityError:
                        st.error("❌ Username or email already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        # ---------------- FORGOT PASSWORD ----------------
        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            if not st.session_state.reset_email:
                email = st.text_input("Registered email address", placeholder="you@infosys.com").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)

                # --- Security Question route ---
                if col_sq.button("Via Security Question", use_container_width=True):
                    # FIX: mandatory-field check (Step 8 names Forgot Password explicitly)
                    if not email:
                        st.error("⚠️ Please enter your registered email address.")
                    # FIX: email-format check (Step 8's email rule applies to every email field, not just Signup)
                    elif not valid_email(email):
                        st.error("❌ Enter a valid email address (e.g. ab@cd.ef).")
                    else:
                        with get_db() as c: r = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                        if r:
                            st.session_state.reset_email = email
                            st.session_state.sq_p = r[0]
                            st.session_state.reset_mode = "sq"
                            st.rerun()
                        else:
                            st.error("❌ Email not found.")

                # --- OTP route ---
                if col_otp.button("Via OTP", use_container_width=True):
                    # FIX: same two checks applied here too
                    if not email:
                        st.error("⚠️ Please enter your registered email address.")
                    elif not valid_email(email):
                        st.error("❌ Enter a valid email address (e.g. ab@cd.ef).")
                    else:
                        with get_db() as c: r = c.execute("SELECT email FROM users WHERE email=?", (email,)).fetchone()
                        if not r:
                            st.error("❌ Email not found.")
                        else:
                            otp = generate_otp()
                            with st.spinner(f"Sending 6-digit OTP (expires in {OTP_EXPIRY_MINUTES} mins)..."):
                                ok, msg = send_otp_email(email, otp)
                            if ok:
                                st.session_state.reset_email = email
                                st.session_state.reset_mode = "otp"
                                st.session_state.otp_jwt = make_otp_token(email, otp)
                                st.session_state.otp_stage = "verify"
                                st.success("✅ OTP sent! Check your inbox.")
                                time.sleep(1)
                                st.rerun()
                            else:
                                st.error(f"❌ {msg}")

            else:
                # --- Security Question flow ---
                if st.session_state.reset_mode == "sq":
                    st.info(f"❓ **Security Question:** {st.session_state.sq_p}")
                    ans = st.text_input("Your answer").lower().strip()
                    npw = st.text_input("New password", type="password", help=PASSWORD_HINT)
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    st.caption(PASSWORD_HINT)
                    st.markdown("<br>", unsafe_allow_html=True)
                    if st.button("Reset Password →", use_container_width=True):
                        # FIX: mandatory-field check on the answer box
                        if not ans:
                            st.error("⚠️ Please enter your security answer.")
                        elif not valid_password(npw):
                            st.error(f"❌ Weak password. {PASSWORD_HINT}")
                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c: r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                            if r and check_txt(ans, r[0]):
            # FIX: block reusing the current password
                              with get_db() as c: cur_pw = c.execute("SELECT password_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                              if cur_pw and check_txt(npw, cur_pw[0]):
                                  st.error("❌ New password cannot be the same as your old password.")
                              else:
                                  with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                  st.success("✅ Password updated successfully!"); time.sleep(1)
                                  st.session_state.reset_email = None; st.session_state.reset_mode = None
                                  navigate("Login")
                            else:
                                st.error("❌ Incorrect security answer.")

                # --- OTP flow ---
                elif st.session_state.reset_mode == "otp":
                    if st.session_state.otp_stage == "verify":
                        st.info(f"📧 Code sent to **{st.session_state.reset_email}** (valid {OTP_EXPIRY_MINUTES} mins).")
                        otp_input = st.text_input("6-digit code", max_chars=6, placeholder="e.g. 849201")
                        st.markdown("<br>", unsafe_allow_html=True)
                        if st.button("Verify Code →", use_container_width=True):
                            # already had a mandatory + format check (6 digits) - unchanged
                            if not otp_input or len(otp_input) != 6:
                                st.error("⚠️ Enter the 6-digit code.")
                            else:
                                ok, msg = verify_otp_token(st.session_state.otp_jwt, otp_input, st.session_state.reset_email)
                                if ok:
                                    st.session_state.otp_stage = "reset"
                                    st.success("✅ Code verified!"); time.sleep(1); st.rerun()
                                else:
                                    st.error(f"❌ {msg}")

                    elif st.session_state.otp_stage == "reset":
                        npw = st.text_input("New password", type="password", help=PASSWORD_HINT)
                        confirm_npw = st.text_input("Confirm new password", type="password")
                        st.caption(PASSWORD_HINT)
                        st.markdown("<br>", unsafe_allow_html=True)
                        if st.button("Update Password →", use_container_width=True):
                            if not valid_password(npw):
                                st.error(f"❌ Weak password. {PASSWORD_HINT}")
                            elif npw != confirm_npw:
                                st.error("❌ Passwords do not match.")
                            else:
                                # FIX: block reusing the current password
                                with get_db() as c: cur_pw = c.execute("SELECT password_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                                if cur_pw and check_txt(npw, cur_pw[0]):
                                    st.error("❌ New password cannot be the same as your old password.")
                                else:
                                    with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                    st.success("🎉 Password updated successfully!"); time.sleep(1)
                                    st.session_state.reset_email = None; st.session_state.reset_mode = None; st.session_state.otp_stage = None
                                    navigate("Login")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                st.session_state.otp_stage = None
                navigate("Login")
# ============================================================
# 9. DASHBOARDS (Admin vs User)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    with get_db() as c: uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]
    is_admin = (email == ADMIN_EMAIL)

    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Infosys Portal</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Panel" if is_admin else "Intelligent Analytics"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if is_admin else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if is_admin else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={"container": {"background-color": COLORS['bg_sidebar']}, "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}})
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    if is_admin:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        # FIX (Step 11): actually list all registered users, no password data
        with get_db() as c:
            rows = c.execute("SELECT username, email FROM users WHERE email != ?", (ADMIN_EMAIL,)).fetchall()

        st.markdown(f"""<div class="pn-card"><h3 style="margin-top:0;">🛡️ Registered Users ({len(rows)})</h3></div>""", unsafe_allow_html=True)
        if rows:
            st.table({"Username": [r[0] for r in rows], "Email": [r[1] for r in rows]})
        else:
            st.info("No registered users yet.")

    else:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)


Overwriting app.py


In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Load ALL secrets from Colab Secrets into the environment
#    (add these 5 secrets in the key icon on the left: JWT_SECRET, NGROK_AUTHTOKEN,
#     EMAIL_ADDRESS, EMAIL_PASSWORD, ADMIN_PASSWORD)
os.environ['JWT_SECRET'] = userdata.get('JWT_SECRET')
os.environ['EMAIL_ADDRESS'] = userdata.get('EMAIL_ADDRESS')
os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
try:
    os.environ['ADMIN_PASSWORD'] = userdata.get('ADMIN_PASSWORD')
except Exception:
    pass  # optional - defaults to admin@123 if not set

NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Kill any existing ngrok tunnels or streamlit sessions
ngrok.kill()
!pkill -f streamlit

# 3. Start Streamlit in the background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 4. Open ngrok tunnel
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🚀 Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n" + "🛑" * 30)
    print("Received Ctrl+C / Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")


🚀 Infosys Portal Live URL: https://wincing-stegosaur-caboose.ngrok-free.dev
⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.
